In [7]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
# from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
import os
from euriai.langchain import create_chat_model
import time

app_dir = os.path.join(os.getcwd(), "app")
load_dotenv(os.path.join(app_dir, ".env"))

api_key = os.getenv("key")

chat_model = create_chat_model(api_key=api_key, model="gpt-4.1-nano", temperature=0.7)
model = chat_model

In [20]:
from euriai.langchain import EuriaiEmbeddings

embeddings = EuriaiEmbeddings(
    api_key=api_key,
    model="text-embedding-3-small"
)

result = embeddings.embed_query("What is machine learning?")
print(len(result))  # 1536 dimensions

1536


In [10]:
prompt = ChatPromptTemplate.from_template("Tell me an interesting fact about {topic}")

prompt_val = prompt.invoke({"topic": "dog"})

print(prompt_val)
print(prompt_val.to_messages())

messages=[HumanMessage(content='Tell me an interesting fact about dog', additional_kwargs={}, response_metadata={})]
[HumanMessage(content='Tell me an interesting fact about dog', additional_kwargs={}, response_metadata={})]


In [17]:
prompt = ChatPromptTemplate.from_template("Tell me an interesting fact about {topic}")
model = model
output_parser = StrOutputParser()
chain = prompt | model | output_parser

chain.invoke({"topic": "dog"})

'Certainly! Did you know that dogs have an extraordinary sense of smell—up to 40 times better than humans? This incredible ability allows them to detect certain diseases, locate missing persons, and even identify specific scents associated with certain conditions like cancer.'

In [18]:
basicchain = model | output_parser
basicchain.invoke("hello!")

'Hello! How can I assist you today?'

### Retrieval Augmented Generation with LCEL

In [26]:
from langchain_core.documents.base import Document
from langchain_chroma import Chroma
from langchain_core.runnables import RunnablePassthrough

embedding_function = EuriaiEmbeddings(api_key=api_key, model="text-embedding-3-small")

docs = [
    Document(
        page_content="the dog loves to eat pizza", metadata={"source": "animal.txt"}
    ),
    Document(
        page_content="the cat loves to eat lasagna", metadata={"source": "animal.txt"}
    ),
]

db = Chroma.from_documents(docs, embedding_function, )
retriever = db.as_retriever()

In [27]:
retriever.invoke("What does the dog want to eat?")

[Document(id='ee9eba81-a5eb-4fbe-b5aa-918e49c2c0e1', metadata={'source': 'animal.txt'}, page_content='the dog loves to eat pizza'),
 Document(id='6e778000-7051-4159-811f-86faf262ecca', metadata={'source': 'animal.txt'}, page_content='the dog loves to eat pizza'),
 Document(id='db4b4c23-05be-43a2-a98d-b3a222ee1d85', metadata={'source': 'animal.txt'}, page_content='the cat loves to eat lasagna'),
 Document(id='55db13f6-3906-440f-bbde-cb71f6079ef9', metadata={'source': 'animal.txt'}, page_content='the cat loves to eat lasagna')]

In [28]:
retriever.invoke("What does the dog want to eat?")

[Document(id='ee9eba81-a5eb-4fbe-b5aa-918e49c2c0e1', metadata={'source': 'animal.txt'}, page_content='the dog loves to eat pizza'),
 Document(id='6e778000-7051-4159-811f-86faf262ecca', metadata={'source': 'animal.txt'}, page_content='the dog loves to eat pizza'),
 Document(id='db4b4c23-05be-43a2-a98d-b3a222ee1d85', metadata={'source': 'animal.txt'}, page_content='the cat loves to eat lasagna'),
 Document(id='55db13f6-3906-440f-bbde-cb71f6079ef9', metadata={'source': 'animal.txt'}, page_content='the cat loves to eat lasagna')]

In [34]:
template = """Answer the question based only on the following context:
{context}

Question: {question}
"""
prompt = ChatPromptTemplate.from_template(template)

model = model
prompt.invoke({
    "context": "Pass your context here",
    "question": "Pass your question here"})

ChatPromptValue(messages=[HumanMessage(content='Answer the question based only on the following context:\nPass your context here\n\nQuestion: Pass your question here\n', additional_kwargs={}, response_metadata={})])

In [46]:
from langchain_core.runnables import RunnablePassthrough, RunnableLambda, RunnableParallel


c = RunnableParallel(
    {
        "context": RunnableLambda(lambda x: x['question']) | retriever,
        "question": RunnableLambda(lambda x: x['question'])
    })

c.invoke({"question": "What does the dog like to eat?"})

{'context': [Document(id='ee9eba81-a5eb-4fbe-b5aa-918e49c2c0e1', metadata={'source': 'animal.txt'}, page_content='the dog loves to eat pizza'),
  Document(id='6e778000-7051-4159-811f-86faf262ecca', metadata={'source': 'animal.txt'}, page_content='the dog loves to eat pizza'),
  Document(id='db4b4c23-05be-43a2-a98d-b3a222ee1d85', metadata={'source': 'animal.txt'}, page_content='the cat loves to eat lasagna'),
  Document(id='55db13f6-3906-440f-bbde-cb71f6079ef9', metadata={'source': 'animal.txt'}, page_content='the cat loves to eat lasagna')],
 'question': 'What does the dog like to eat?'}

In [47]:
retrieval_chain = (
    RunnableParallel(
    {
        "context": RunnableLambda(lambda x: x['question']) | retriever,
        "question": RunnableLambda(lambda x: x['question'])
    })            
    | prompt
    | model
    | StrOutputParser()
)

retrieval_chain.invoke({"question": "What does the dog like to eat?"})

'The dog likes to eat pizza.'

In [49]:
template = """Answer the question based only on the following context:
{context}

Question: {question}
"""
prompt = ChatPromptTemplate.from_template(template)
model = model

retrieval_chain = (
    RunnableParallel({"context": retriever, "question": RunnablePassthrough()})
    | prompt
    | model
    | StrOutputParser()
)

In [50]:
retrieval_chain.invoke("What does the dog like to eat?")

'The dog loves to eat pizza.'